---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-41: Model Context Protocol - I</h1>

# Learning agenda of this notebook

1. Agentic Protocols
2. Issue with Tools Usage in Traditional Systems
3. Architecture of Model Context Protocol (MCP)
4. Primitives of Model Context Protocol (MCP)
5. MCP Client-Server Communication
6. MCP Transport Mechanisms (Local vs Remote)

# <span style='background :lightgreen' >1. Agentic Protocols</span>
<h1 align="center"><div class="alert alert-success" style="margin: 20px">Agentic protocols are standardized communication and control mechanisms that allow LLMs, Tools, Memory systems and other agents to interact reliably, safely, and predictably</h1>


<div style="text-align:center;">
    <img src="../images/agenticai4.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

<h2 align="center"><div class="alert alert-success" style="margin: 20px">MCP is a protocol introduced by Anthropic that standardizes how external tools, data, and memory are exposed to an LLM through a client-server architecture using JSON-RPC</h2>

<h2 align="center"><div class="alert alert-success" style="margin: 20px">Agent-to-Agent (A2A) is a protocol that enables multiple AI agents from different vendors and frameworks to securely discover, communicate and collaborate with each other through JSON-RPC messaging over HTTP</h2>


<img src="../images/agenticai5.png" style="float: right; margin-left: 20px; margin-bottom: 10px;" width="1000">

# <span style='background :lightgreen' >2. Issue with Tools Usage in Traditional Systems</span>
### (i) NxM Integration Problem
<h3 align="center"><div class="alert alert-success" style="margin: 20px">If you have N different AI applications/Models (GPT, Claude, Llama) and M different tools/systems (GitHub, Slack, databases, etc.), you needed to build N×M different integrations</h3>

### (ii) Each Model Provider has its own Tool Schema
- OpenAI: https://platform.openai.com/docs/guides/function-calling
- Gemini: https://ai.google.dev/gemini-api/docs/function-calling
- Claude: https://docs.claude.com/en/docs/agents-and-tools/tool-use/overview
- Qwen: https://www.alibabacloud.com/help/en/model-studio/qwen-function-calling
- Gema: https://ai.google.dev/gemma/docs/capabilities/function-calling

<h3 align="center"><div class="alert alert-success" style="margin: 20px">Models are only as good as the Context given to them</h3>
    
### Scenario
- **Traditional Systems (old way):** Suppose inside an LLM app needs to connect to Postgres and SQlite databases. For Postgres, developers will use `psycopg2`, while for SQLite the developers will use `sqlite3`. Without MCP, the developers will have to write custom code for both separately, and the LLM will make separate, database-specific calls for each database.
- **Using MCP (new way)** Developers will use the MCP Server for Postgres and the MCP Server for SQLite, and they both have a standardized query_database command. Claude now only needs to send the standardized query_database command  to query both databases.

## Solution: Model Context Protocol (MCP)


<img src="../images/mcp-usb.png" style="float: right; margin-left: 20px; margin-bottom: 10px;" width="800">


<img src="../images/mcp5.png" style="float: right; margin-left: 20px; margin-bottom: 10px;" width="400">

- MCP (https://modelcontextprotocol.io/docs/getting-started/intro) was introduced by Anthropic in Nov 2024. It is a standard way (a protocol) that lets AI systems (LLMs, agents, etc.) to interact with external tools, apps, data, and services without custom wiring for each tool. It simplifies creating connections across different APIs without individual custom scripts.
>- MCP makes it extremely easy for developers to connect to thousands of different external systems and tools, provided:
>    - The host support the MCP client
>    - The external system has an MCP server


- **Analogy: USB hub for AI tools:** Think of the **MCP Client** like a **USB hub**.  
    - Plug different “devices” (MCP Servers) into the hub.  
    - Each server exposes one or more tools/resources.  
    - The host app just talks to the hub in one standard way.
    - Think of MCP like a USB-C port for AI applications—just as USB-C provides a standardized way to connect electronic devices, MCP provides a standardized way to connect AI applications to external systems 

# <span style='background :lightgreen' >3. Architecture of Model Context Protocol (MCP)</span>

<div style="text-align:center;">
    <img src="../images/agenticai6.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

<img src="../images/mcp-architecture.png" style="float: right; margin-left: 20px; margin-bottom: 10px;" width="1000">

- **MCP Host:**  An application that contains an MCP client and provides the user interface. The host runs the LLM, manages the UI, and orchestrates communication with MCP servers. It can spawn local servers (running on your machine) or connect to remote servers (hosted on the internet). Some popular MCP Hosts (IDEs & Applications) are:
    - **Claude Desktop:**  A GUI chat interface by Anthropic that support MCP (https://claude.ai/download)
    - **Google Gemini CLI:** A CLI chat interface for Google's Gemini with MCP support
    - **Claude Code:**  AI coding assistant CLI tool
    - **VS Code:** Microsoft's code editor with MCP support (https://code.visualstudio.com/)
    - **GitHub Copilot:**  An AI assistant extension/plugin that runs inside VS Code (and other editors). (https://github.com/features/copilot)
    - **Cursor:** AI-first code editor with native MCP (https://cursor.com/)
    - **Trae:** An AI-powered IDE with built-in MCP client support and a one-click marketplace for MCP servers. (https://www.trae.ai/)
- **MCP Client:**  The software component (connector) embedded in the Host that speaks MCP (JavaScript Object Notation - Remote Procedure Call (JSON-RPC)) to handle communication between MCP host and MCP servers. MCP client performs initialization, discovery, tool calls, subscriptions, and shutdown for a single session. A Host can run many clients (one per server or per user context).
    - **What the MCP Client Does:**
        - Sends your requests (like "list files" or "search GitHub") to the appropriate MCP server
        - Receives results from servers and displays them in your application
        - Handles authentication (API keys, OAuth tokens) automatically
        - Manages connections to multiple servers simultaneously (e.g., one for files, one for GitHub)
        - Performs initialization, tool discovery, execution, and shutdown for each session
    - **Simple Example:** When you type "What issues are open in my repo?" in Claude Desktop, the MCP Client:
        1. Sends the request to the GitHub MCP Server
        2. Waits for the server to fetch the data
        3. Receives the list of issues
        4. Displays them in the chat interface
- **MCP Server:** An MCP Server is a service (local or remote) that provides specific tools and capabilities to MCP clients. Servers execute the actual work—calling APIs, querying databases, processing files, etc. Some key characteristics are:
    - Each server connects to specific tools, apps, or data sources
    - Designed to be modular (minimal dependencies, easy to add/remove)
    - Can expose multiple tools (e.g., Airbnb server, Slack server).
    - Interfaces with external APIs (Slack, Dropbox, Gmail) or local computations.
- **MCP Protocol:** Standardized communication layer between MCP clients and MCP servers. Defines the transport and data exchange process. The Transport layer of MCP protocol controls how the messages flow between client and server. For example, `stdio` for local (spawned) servers, and `HTTP /WebSocket/SSE` for remote servers. MCP messages are JSON-RPC 2.0 payloads wrapped in the chosen transport. The transport determines connection lifecycle behavior (e.g., how shutdown/close works).

>- RPC allows a program to execute a function on another computer as if it were local, hiding the details of network communication and data transfer. This abstraction makes it easier to build distributed applications.
>- JSON-RPC combines the concepts of RPC with the simplicity of JSON, allowing developers to structure RPC requests and responses in a standardized JSON format.
>- Unlike REST, JSON-RPC supports batching (multiple requests at once) and notifications (fire-and-forget calls).
>- JSON-RPC can enable bi-directional communication when used over a persistent connection like WebSocket.

# <span style='background :lightgreen' >4. Primitives of Model Context Protocol (MCP)</span>

## a. MCP Clients can ask for following Features from MCP Servers

<div style="text-align:center;">
    <img src="../images/mcp-99.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

- An MCP server can contain three types of primitives:
    - **Tools** (https://modelcontextprotocol.io/specification/2025-06-18/server/tools) tools are actions (functions, APIs, etc) the AI ask the MCP server to perform. For example a GitHub MCP server might have tools like `createIssue`, `getRepoContent`, `listBranches`, etc.  A Google Drive MCP server might have tools that can search your Google Drive for specifice files, or can create new files inside your Google Drive. Similarly, a local MCP server might run local computations (math, regression, screenshot automation). Tools are controlled by the Model (LM), which decides *when*, *how*, and *with what arguments* to call a tool. Tools are perfect for giving your AI model additional capabilities it can use autonomously. When you ask Claude to "calculate the square root of 3 using JavaScript," it's Claude that decides to use a JavaScript execution tool to run the calculation. The key property of a tool is that it performs an action (not just data retrieval). In MCP, each tool must define Name (e.g., `get_weather`), Description (explains  what the toold does and the arguments it expects) and Input Schema (dictionary of arguments the tool accepts).  To deal with tools, MCP server normally provides two functions:
        - `tools/list` MCP client ask the MCP server "What tools do you provide?"
        - `tools/calls` MCP client tells the MCP server "Please run this tool with these arguments"
    - **Resources:** (https://modelcontextprotocol.io/specification/2025-06-18/server/resources) Resources are structured data sources (static context and data in string, JSON or binary format), for the user or the AI model to use. Resources are controlled by your application code. Your app decides when to fetch resource data and how to use it - typically to add context to conversations. For example, the MCP host/client if connected to GitHub MCP server can request it to fetch the README file of a specific GitHub repository. Similarly, if the MCP client if connected to some MCP Database server can request it to fetch the database schema of a specific database. The purpose of a resource is that it supplies background data (**context**) rather than executing actions. (A resourece can be a direct resource or templated resource). To deal with resources, MCP server normally provides following functions:
        - `resources/list` MCP client ask the MCP server "What resources are available?"
        - `resources/read` MCP client ask the MCP server "Give me the content of this resource"
        - `resources/subscribe` MCP client subscribes for updates, which means client can ask to get updates when resource changes
        - `resources/unsubscribe` MCP client unsubscribes from updates
    - **Prompt Templates:** (https://modelcontextprotocol.io/specification/2025-06-18/server/prompts) Prompts are predefined prompt templates that the MCP server offers that client/model can use. The purpose of a prompt is that it helps users apply effective, pre-crafted LLM instructions. For example, scientific research report templates, literature review prompts and report-writing scaffolds. The workflow of using prompt is first the user selects a prompt which replaces/augments the user's input and the LLM is called with that structured prompt for better results. To deal with prompts, MCP server normally provides two functions:
        - `prompts/list` MCP client ask the MCP server "What prompt templates do you provide?"
        - `prompts/get` MCP client fetches a specific prompt template

## b. MCP Servers can ask for following Features from MCP Clients
- **Sampling:** (https://modelcontextprotocol.io/specification/2025-06-18/client/sampling) MCP server ask the client to ask AI for for his intelligence. An example of a **GitHub MCP server** can be:
```
Server → Client: "I found 50 changed files. Can you ask the AI to write a good commit message summarizing these changes?"
Client → AI Model: "Write a commit message for these changes..."
AI Model → Client: "commit message....."
Client → Server: Here's what the AI said: "commit message..."
```
- **Roots:** (https://modelcontextprotocol.io/specification/2025-06-18/client/roots) MCP server ask the client to check the configuration and inform about the filesystem boundaries to operate in. An example of a **File MCP server** can be:
```
Server → Client: "Where am I allowed to work? Which folders can I access?"
Client → Server: "You can access: /home/user/projects/ and /home/user/documents/"
Server: (Now knows it can only read/write files in those two locations)
```
- **Elicitation:** (https://modelcontextprotocol.io/specification/2025-06-18/client/elicitation) MCP server ask the client to get some additional information from the human user. An example of a **Database MCP server** can be:
```
Server → Client: "I need the database password to connect. Can you ask the user?"
Client → User: "The database server needs your password"
User types: "mySecurePassword123"
Client → Server: "The user said the password is: mySecurePassword123"
```

# <span style='background :lightgreen' >5. MCP Client-Server Communication</span>
<img src="../images/how-mcp-work.png" style="float: right; margin-left: 20px; margin-bottom: 10px;" width="1300">

#### The user submits his/her Query to your server/application:  "What repositories do I have?"
- **Tool Discovery:** Your server/application needs to know what tools are available to send to Claude.
- **List Tools Exchange:** Your server/application asks the MCP client for available tools.
- **MCP Communication:** The MCP client sends a `ListToolsRequest` to the MCP server and receives a `ListToolsResult`.
- **User Query:** Your server/application sends the user's query plus the available tools to Claude.
- **Tool Use Decision:** Claude decides whether it needs to call a tool to answer the question or not (we assume a yes)
- **Tool Execution Request:** Your server/application asks the MCP client to run the tool Claude specified.
- **External API Call:** The MCP client sends a `CallToolRequest` to the MCP server, which makes the actual GitHub API call.
- **Results Flow Back:** GitHub responds with repository data, which flows back through the MCP server as a `CallToolResult`.
- **Tool Result to Claude:** Your server/application sends the tool results back to Claude.
- **LLM Response Generation:** The LLM processes the tool result and generates a natural, user-friendly response in appropriate format and sends the response back to your server/application via MCP client.
- **User Gets Answer:** The MCP Client displays the LLM's response to the user via your server/application

# <span style='background :lightgreen' >6. MCP Transport Mechanisms (Local vs Remote) https://modelcontextprotocol.io/specification/2025-06-18/basic/transports</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">The MCP Host and Client usually run on local machine (desktop apps), while the MCP servers can run locally as well as on a remote VM-based machine accessible via the network.</h3>
<h3 align="center"><div class="alert alert-success" style="margin: 20px">An MCP server—whether running locally or remotely—can handle local computations (e.g., mathematical functions, linear regression, capturing and describing screenshots with an LLM, organizing desktop files, or performing intelligent searches over the local file system) as well as invoke external APIs such as GitHub, Gmail, Dropbox, or Slack.</h3>

## a. Modes of Transport used by Local and Remote MCP Servers

<img align="right" width="550" src="../images/local-mcpserver.png"  >

### (i) Local MCP Server (Stdio Protocol): 
- A local MCP server runs on the same machine as the MCP client (i.e., on your local computer) and can access both local resources (files, databases, applications) and remote APIs (Google Drive, GitHub etc).
- Data processing happens on your machine. Require installation and configuration via `claude_desktop_config.json` or similar config files.
- Use stdio transport protocol, when building tools that need to interact with the local environment (e.g., file system, processes, screen automation)..
- Typically, each client instance starts its own server process (one per user).
- **How it works:**
    - The MCP client/host is told how to download and run the server (via a config script or package, e.g., `npm`, `npx`, or `uvx`).
    - The MCP client installs and launches the MCP server as a subprocess on the same machine.
    - The client writes JSON-RPC messages to the server’s stdin.
    - The server processes the messages and replies via its stdout. (May or may not access remote services)
- **Benefits:**
    - Very fast communication (direct inter-process communication).
    - More secure — no open network port exposed.
    - Required for tools that access local resources (file system, desktop automation, local search).
    - Easy to set up, no network or extra dependencies.
- **Limitations:**
    - Code updates require each user to reinstall or refresh locally.
    - Computations run on the local machine, which can be resource-heavy.
- **Examples of Local MCP Servers:**
    - Filesystem MCP Server (Access and manage files on your computer): https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem 
    - Git MCP Server (Interact with local Git repositories): https://github.com/modelcontextprotocol/servers/tree/main/src/git
    - GitHub MCP Server (Interacts with GitHub APIs requiring GitHub PATs): https://github.com/modelcontextprotocol/servers-archived/tree/main/src/github
    - Google Drive MCP Server (Access Google Drive with Google OAuth credentials): https://github.com/modelcontextprotocol/servers/tree/main/src/gdrive
    - Slack MCP Server (Local server for Slack interactions): https://github.com/modelcontextprotocol/servers/tree/main/src/slack, https://github.com/zencoderai/slack-mcp-server
    - Fetch MCP Server (Search the web): https://github.com/modelcontextprotocol/servers/tree/main/src/fetch
    - A Twitter MCP Server: https://github.com/EnesCinr/twitter-mcp

<br><br><br>

### (ii) Remote MCP Server(HTTP+SSE and Streamable HTTP Protocol)

<img align="right" width="550" src="../images/remote-mcpserver.png"  >

- A remote MCP server is hosted on the internet rather than your local machine.
- No need to install or run locally, and updates are applied automatically.
- Internet-accessible with simplified sign-in and permission granting.
- You can use either of the two protocols:
    - For stateful connection, use HTTP+SSE (Server Sent Events) that came in 05-11-2024
    - For stateful as well as stateless connection, use Streamable HTTP that came in 26-03-2025
- **How it works:**
    - The MCP client/host is given the server URL instead of a local install path.
    - The client communicates with the server using JSON-RPC over HTTP POST requests (with JSON payloads).
    - The server can also send events/updates using Server-Sent Events (SSE).
    - Authentication is typically handled via API keys and typically use OAuth for easier setup
- **Benefits:**
    - Updating the server automatically updates all clients.
    - A single remote server can serve many MCP clients at once.
    - Offloads heavy computations (ML inference, image generation, data processing).
    - More scalable and portable — clients only need the server URL.
    - Enables web-based MCP clients (e.g., ChatGPT or future browser apps).
- **Limitations:**
    - As of early 2025, HTTP support is still limited in some MCP hosts.
    - Slightly slower due to network latency.
## b. Where to Get MCP Servers?
- **Reference Servers (Official):**
    - These servers are intended as reference implementations to demonstrate MCP features and SDK usage — they are meant to serve as educational examples for developers building their own MCP servers, not as production-ready solutions.
    - Published to npm as scoped packages under `@modelcontextprotocol/`.
    - Installation: Use `npx -y @modelcontextprotocol/server-[name]` in your config. NPX automatically downloads from npm, caches locally, and runs directly (no manual cloning needed).
    - Examples:
        - Filesystem: `@modelcontextprotocol/server-filesystem` — https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem
        - Fetch: `@modelcontextprotocol/server-fetch` — https://github.com/modelcontextprotocol/servers/tree/main/src/fetch
        - Git: `@modelcontextprotocol/server-git` — https://github.com/modelcontextprotocol/servers/tree/main/src/git
- **Community MCP Servers:** *(Community-curated collections: https://github.com/punkpeye/awesome-mcp-servers, https://mcpservers.org/, https://mcp-get.com/)*
    - A growing set of community-developed and maintained servers demonstrating various MCP applications across different domains.
    - ⚠️ Community servers are untested and should be used at your own risk. They are not affiliated with or endorsed by Anthropic.
    - Most MCP servers are built in Node.js/TypeScript or Python, though some exist in Go or Rust.
    - Publishing status varies widely:
        - Some are published as npm packages: `npx -y package-name` (no cloning needed)
        - Some are published as PyPI packages: `uvx package-name` or `pip install package-name`
        - Some exist only as GitHub source code: requires manual cloning and absolute file paths in config
        - Some are available as Docker containers: `docker run -i --rm image-name`
        - Some are available as Desktop Extensions (`.mcpb`): drag-and-drop installation
- **Official Integrations** (company-maintained, production-ready servers — hosted in each company's own repo):
    - These are MCP servers built and maintained by companies for their own platforms.
    - Most can be added inside Claude Desktop via Connectors (one-click), some via the config file, and some support both.
    - Examples:
        - GitHub: https://github.com/github/github-mcp-server *(official Go-based server — the old `@modelcontextprotocol/server-github` npm package was archived in 2025)*
        - Slack: maintained by Zencoder — https://github.com/zencoderai/slack-mcp-server
        - Atlassian: https://www.atlassian.com/platform/remote-mcp-server
        - Stripe: https://mcp.stripe.com
        - PayPal: https://mcp.paypal.com/mcp

<h1 align="center"><div class="alert alert-info" style="margin: 20px; background-color: #cce5ff; border-color: #b8daff; color: #004085;"><b>We will write our own MCP Servers and our own MCP Clients but to start with let us use existing MCP Clients (Claude Desktop) and existing MCP Servers</b></div></h1>